In [7]:
from collections import defaultdict

ALPHABET = ['A', 'C', 'G', 'T']
INF = 10**9

# ---------------------------------------------------------
# 解析输入（适配：内部节点是数字，叶子是 DNA 字符串）
# ---------------------------------------------------------
def parse_input_small_parsimony(lines):
    n = int(lines[0].strip())
    children = defaultdict(list)
    parent = {}
    label = {}      # 叶子节点的 DNA
    all_nodes = set()

    for line in lines[1:]:
        if not line.strip():
            continue
        u, v = line.strip().split("->")
        u = u.strip()
        v = v.strip()

        # u 一定是内部节点（数字）
        if u.isdigit():
            u_id = int(u)
        else:
            u_id = u  # 理论上不会发生
        all_nodes.add(u_id)

        if v.isdigit():
            # 内部节点
            v_id = int(v)
            children[u_id].append(v_id)
            parent[v_id] = u_id
            all_nodes.add(v_id)
        else:
            # 叶子：v 本身就是节点 ID，也是 DNA 序列
            leaf_id = v
            children[u_id].append(leaf_id)
            parent[leaf_id] = u_id
            all_nodes.add(leaf_id)
            label[leaf_id] = leaf_id  # 叶子的标签就是它自己

    # 找根：没有父节点的那个内部节点
    root = None
    for v in all_nodes:
        if v not in parent:
            root = v
            break

    return n, root, children, parent, label, all_nodes


# ---------------------------------------------------------
# 后序遍历
# ---------------------------------------------------------
def postorder(root, children):
    order = []
    def dfs(v):
        for c in children.get(v, []):
            dfs(c)
        order.append(v)
    dfs(root)
    return order


# ---------------------------------------------------------
# Small Parsimony（单列）
# ---------------------------------------------------------
def small_parsimony_single_column(root, children, leaf_char, all_nodes):
    score = {v: {k: INF for k in ALPHABET} for v in all_nodes}
    order = postorder(root, children)

    # bottom-up
    for v in order:
        if v in leaf_char:
            # 叶子
            true = leaf_char[v]
            for k in ALPHABET:
                score[v][k] = 0 if k == true else INF
        else:
            # 内部节点
            left, right = children[v]
            for k in ALPHABET:
                best_left = min(score[left][i] + (0 if i == k else 1) for i in ALPHABET)
                best_right = min(score[right][j] + (0 if j == k else 1) for j in ALPHABET)
                score[v][k] = best_left + best_right

    # top-down 回溯
    char_choice = {}
    root_char = min(ALPHABET, key=lambda k: score[root][k])
    char_choice[root] = root_char
    column_score = score[root][root_char]

    def assign(v):
        if v not in children:
            return
        pchar = char_choice[v]
        for c in children[v]:
            best = min(ALPHABET, key=lambda i: score[c][i] + (0 if i == pchar else 1))
            char_choice[c] = best
            assign(c)

    assign(root)
    return char_choice, column_score


# ---------------------------------------------------------
# Small Parsimony（全序列）
# ---------------------------------------------------------
def small_parsimony_full(lines):
    n, root, children, parent, label, all_nodes = parse_input_small_parsimony(lines)

    # 所有叶子长度相同
    any_leaf = next(iter(label.values()))
    m = len(any_leaf)

    node_chars = {v: [''] * m for v in all_nodes}
    total_score = 0

    for pos in range(m):
        leaf_char = {v: s[pos] for v, s in label.items()}
        char_choice, col_score = small_parsimony_single_column(
            root, children, leaf_char, all_nodes
        )
        total_score += col_score
        for v, ch in char_choice.items():
            node_chars[v][pos] = ch

    # 拼接成完整 DNA
    full_label = {v: ''.join(chars) for v, chars in node_chars.items()}

    # 输出
    out = []
    out.append(str(total_score))
    for u in children:
        for v in children[u]:
            s1 = full_label[u]
            s2 = full_label[v]
            dist = sum(a != b for a, b in zip(s1, s2))
            out.append(f"{s1}->{s2}:{dist}")
            out.append(f"{s2}->{s1}:{dist}")
    return "\n".join(out)


In [10]:
input_text = """4
4->CAAATCCC
4->ATTGCGAC
5->CTGCGCTG
5->ATGGACGA
6->4
6->5
"""

lines = input_text.strip().split("\n")
print(small_parsimony_full(lines))


16
ATAGACAC->CAAATCCC:5
CAAATCCC->ATAGACAC:5
ATAGACAC->ATTGCGAC:3
ATTGCGAC->ATAGACAC:3
ATGGACAA->CTGCGCTG:5
CTGCGCTG->ATGGACAA:5
ATGGACAA->ATGGACGA:1
ATGGACGA->ATGGACAA:1
ATAGACAA->ATAGACAC:1
ATAGACAC->ATAGACAA:1
ATAGACAA->ATGGACAA:1
ATGGACAA->ATAGACAA:1


In [65]:
from collections import defaultdict

ALPHABET = ['A', 'C', 'G', 'T']
INF = 10**9

def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))

# ---------------- Parsing ----------------

def parse_input(lines):
    n = int(lines[0].strip())
    children = defaultdict(set)
    leaf_label = {}

    for line in lines[1:]:
        if not line.strip():
            continue
        u, v = line.strip().split("->")
        u = u.strip()
        v = v.strip()

        def parse_node(x):
            if all(c in "ACGT" for c in x):
                leaf_label[x] = x
                return x
            else:
                return int(x)

        u_id = parse_node(u)
        v_id = parse_node(v)

        children[u_id].add(v_id)
        children[v_id].add(u_id)

    return n, children, leaf_label

# ---------------- Rooted helpers ----------------

def build_rooted_children(root, children_undirected):
    children_down = defaultdict(list)
    parent = {}

    def dfs(v, p=None):
        parent[v] = p
        for c in children_undirected[v]:
            if c != p:
                children_down[v].append(c)
                dfs(c, v)

    dfs(root)
    return children_down, parent

def postorder(root, children_down):
    order = []
    def dfs(v):
        for c in children_down[v]:
            dfs(c)
        order.append(v)
    dfs(root)
    return order

# ---------------- Small Parsimony (single column) ----------------

def small_parsimony_single_column(root, children_down, leaf_char):
    order = postorder(root, children_down)
    score = {v: {k: INF for k in ALPHABET} for v in children_down}

    # bottom-up
    for v in order:
        if v in leaf_char:
            true = leaf_char[v]
            for k in ALPHABET:
                score[v][k] = 0 if k == true else INF
        else:
            for k in ALPHABET:
                score[v][k] = sum(
                    min(score[c][x] + (x != k) for x in ALPHABET)
                    for c in children_down[v]
                )

    # top-down
    char_choice = {}
    root_char = min(ALPHABET, key=lambda k: score[root][k])
    char_choice[root] = root_char
    total_score = score[root][root_char]

    def assign(v):
        p = char_choice[v]
        for c in children_down[v]:
            best = min(ALPHABET, key=lambda x: score[c][x] + (x != p))
            char_choice[c] = best
            assign(c)

    assign(root)
    return char_choice, total_score

# ---------------- Small Parsimony (unrooted) ----------------

def small_parsimony_unrooted(children_undirected, leaf_label):
    # 选一个内部节点作为 root（整数）
    root = next(v for v in children_undirected if isinstance(v, int))

    children_down, parent = build_rooted_children(root, children_undirected)

    m = len(next(iter(leaf_label.values())))
    all_nodes = set(children_undirected.keys())
    node_chars = {v: [''] * m for v in all_nodes}
    total_score = 0

    for pos in range(m):
        leaf_char = {node: s[pos] for node, s in leaf_label.items()}
        char_choice, col_score = small_parsimony_single_column(root, children_down, leaf_char)
        total_score += col_score
        for node, ch in char_choice.items():
            node_chars[node][pos] = ch

    full_label = {v: ''.join(node_chars[v]) for v in node_chars}
    return total_score, full_label

# ---------------- Formatting output ----------------

def format_tree(children, full_label):
    edges = set()
    for u in children:
        for v in children[u]:
            if (v, u) not in edges:
                edges.add((u, v))

    # 为了稳定输出，按标签排序
    edges = sorted(edges, key=lambda e: (full_label[e[0]], full_label[e[1]]))

    out = []
    for u, v in edges:
        su = full_label[u]
        sv = full_label[v]
        d = hamming(su, sv)
        out.append(f"{su}->{sv}:{d}")
        out.append(f"{sv}->{su}:{d}")
    return "\n".join(out)

# ---------------- Main wrapper ----------------

def solve_small_parsimony_unrooted(text):
    lines = text.strip().split("\n")
    n, children, leaf_label = parse_input(lines)
    score, full_label = small_parsimony_unrooted(children, leaf_label)
    result = str(score) + "\n" + format_tree(children, full_label)
    return result


In [66]:
input_text = """4
TCGGCCAA->4
4->TCGGCCAA
CCTGGCTG->4
4->CCTGGCTG
CACAGGAT->5
5->CACAGGAT
TGAGTACC->5
5->TGAGTACC
4->5
5->4
"""

print(solve_small_parsimony_unrooted(input_text))


17
CAAGGAAA->TGAGTACC:5
TGAGTACC->CAAGGAAA:5
CACAGGAT->CAAGGAAA:4
CAAGGAAA->CACAGGAT:4
CCAGGCAA->CAAGGAAA:2
CAAGGAAA->CCAGGCAA:2
CCAGGCAA->CCTGGCTG:3
CCTGGCTG->CCAGGCAA:3
TCGGCCAA->CCAGGCAA:3
CCAGGCAA->TCGGCCAA:3


In [67]:
from collections import defaultdict
import copy

# ---------------------------------------------------------
# Parse undirected tree
# ---------------------------------------------------------
def parse_tree(lines):
    a, b = lines[0].split()
    a = int(a)
    b = int(b)

    children = defaultdict(set)
    for line in lines[1:]:
        if not line.strip():
            continue
        u, v = line.strip().split("->")
        u = u.strip()
        v = v.strip()

        # convert to int if internal node
        if u.isdigit():
            u = int(u)
        if v.isdigit():
            v = int(v)

        children[u].add(v)
        children[v].add(u)

    return a, b, children


# ---------------------------------------------------------
# Find the two neighbors of edge (a, b)
# ---------------------------------------------------------
def nearest_neighbors(a, b, children):
    # children of a except b
    a_neighbors = [x for x in children[a] if x != b]
    # children of b except a
    b_neighbors = [x for x in children[b] if x != a]

    x1, x2 = a_neighbors
    y1, y2 = b_neighbors

    # Two NNI swaps:
    # 1. swap x2 <-> y1
    # 2. swap x2 <-> y2

    def build_neighbor(swap_x, swap_y):
        T = copy.deepcopy(children)

        # detach
        T[a].remove(swap_x)
        T[swap_x].remove(a)

        T[b].remove(swap_y)
        T[swap_y].remove(b)

        # attach swapped
        T[a].add(swap_y)
        T[swap_y].add(a)

        T[b].add(swap_x)
        T[swap_x].add(b)

        return T

    T1 = build_neighbor(x2, y1)
    T2 = build_neighbor(x2, y2)

    return T1, T2


# ---------------------------------------------------------
# Format adjacency list
# ---------------------------------------------------------
def format_adj(children):
    out = []
    for u in children:
        for v in children[u]:
            out.append(f"{u}->{v}")
    return "\n".join(out)


# ---------------------------------------------------------
# Main function
# ---------------------------------------------------------
def solve_nearest_neighbors(lines):
    a, b, children = parse_tree(lines)
    T1, T2 = nearest_neighbors(a, b, children)

    return format_adj(T1) + "\n\n" + format_adj(T2)

In [68]:
input_text = """5 4
0->4
4->0
1->4
4->1
2->5
5->2
3->5
5->3
4->5
5->4"""

lines = input_text.strip().split("\n")
print(solve_nearest_neighbors(lines))


0->5
4->1
4->3
4->5
1->4
2->5
5->0
5->2
5->4
3->4

0->4
4->0
4->3
4->5
1->5
2->5
5->1
5->2
5->4
3->4


In [57]:
from collections import defaultdict
import copy

ALPHABET = ['A','C','G','T']
INF = 10**9

def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))

# ---------------------------------------------------------
# Parse input
# ---------------------------------------------------------
def parse_input(lines):
    n = int(lines[0].strip())
    children = defaultdict(set)
    leaf_label = {}

    for line in lines[1:]:
        if not line.strip():
            continue
        u, v = line.split("->")
        u = u.strip()
        v = v.strip()

        def parse_node(x):
            if all(c in "ACGT" for c in x):
                leaf_label[x] = x
                return x
            else:
                return int(x)

        u_id = parse_node(u)
        v_id = parse_node(v)

        children[u_id].add(v_id)
        children[v_id].add(u_id)

    return n, children, leaf_label


# ---------------------------------------------------------
# Build rooted tree
# ---------------------------------------------------------
def build_rooted_children(root, children_undirected):
    children_down = defaultdict(list)
    parent = {}

    def dfs(v, p=None):
        parent[v] = p
        for c in children_undirected[v]:
            if c != p:
                children_down[v].append(c)
                dfs(c, v)

    dfs(root)
    return children_down, parent


def postorder(root, children_down):
    order = []
    def dfs(v):
        for c in children_down[v]:
            dfs(c)
        order.append(v)
    dfs(root)
    return order


# ---------------------------------------------------------
# Small Parsimony (single column)
# ---------------------------------------------------------
def small_parsimony_single_column(root, children_down, leaf_char):
    order = postorder(root, children_down)
    score = {v: {k: INF for k in ALPHABET} for v in children_down}

    # bottom-up
    for v in order:
        if v in leaf_char:
            true = leaf_char[v]
            for k in ALPHABET:
                score[v][k] = 0 if k == true else INF
        else:
            for k in ALPHABET:
                score[v][k] = sum(
                    min(score[c][x] + (x != k) for x in ALPHABET)
                    for c in children_down[v]
                )

    # top-down
    char_choice = {}
    root_char = min(ALPHABET, key=lambda k: score[root][k])
    char_choice[root] = root_char
    total_score = score[root][root_char]

    def assign(v):
        p = char_choice[v]
        for c in children_down[v]:
            best = min(ALPHABET, key=lambda x: score[c][x] + (x != p))
            char_choice[c] = best
            assign(c)

    assign(root)
    return char_choice, total_score


# ---------------------------------------------------------
# Small Parsimony (full sequences)
# ---------------------------------------------------------
def small_parsimony_unrooted(children_undirected, leaf_label):
    root = next(v for v in children_undirected if isinstance(v, int))
    children_down, parent = build_rooted_children(root, children_undirected)

    m = len(next(iter(leaf_label.values())))
    all_nodes = set(children_undirected.keys())
    node_chars = {v: [''] * m for v in all_nodes}
    total_score = 0

    for pos in range(m):
        leaf_char = {node: s[pos] for node, s in leaf_label.items()}
        char_choice, col_score = small_parsimony_single_column(root, children_down, leaf_char)
        total_score += col_score
        for node, ch in char_choice.items():
            node_chars[node][pos] = ch

    full_label = {v: ''.join(node_chars[v]) for v in node_chars}
    return total_score, full_label


# ---------------------------------------------------------
# NNI
# ---------------------------------------------------------
def internal_edges(children):
    edges = set()
    for u in children:
        if len(children[u]) == 1:
            continue
        for v in children[u]:
            if len(children[v]) == 1:
                continue
            if (v, u) not in edges:
                edges.add((u, v))
    return edges


def nni_neighbors_on_edge(children, a, b):
    a_neighbors = [x for x in children[a] if x != b]
    b_neighbors = [x for x in children[b] if x != a]

    if len(a_neighbors) != 2 or len(b_neighbors) != 2:
        return []

    x1, x2 = a_neighbors
    y1, y2 = b_neighbors

    def build_neighbor(swap_x, swap_y):
        T = copy.deepcopy(children)
        T[a].remove(swap_x)
        T[swap_x].remove(a)
        T[b].remove(swap_y)
        T[swap_y].remove(b)
        T[a].add(swap_y)
        T[swap_y].add(a)
        T[b].add(swap_x)
        T[swap_x].add(b)
        return T

    return [
        build_neighbor(x2, y1),
        build_neighbor(x2, y2)
    ]


# ---------------------------------------------------------
# Printing (Cogniterra format)
# ---------------------------------------------------------
def format_tree(children, full_label):
    edges = []
    for u in children:
        for v in children[u]:
            edges.append((full_label[u], full_label[v]))

    edges = sorted(edges)
    out = []
    for u, v in edges:
        out.append(f"{u}->{v}:{hamming(u, v)}")
    return "\n".join(out)


# ---------------------------------------------------------
# NNI Heuristic
# ---------------------------------------------------------
def nearest_neighbor_interchange(n, children, leaf_label):
    score, full_label = small_parsimony_unrooted(children, leaf_label)
    current_score = score
    current_children = children
    current_label = full_label

    results = []
    results.append(str(current_score) + "\n" + format_tree(current_children, current_label))

    while True:
        best_score = current_score
        best_children = None
        best_label = None

        for (a, b) in internal_edges(current_children):
            for neigh in nni_neighbors_on_edge(current_children, a, b):
                s, lab = small_parsimony_unrooted(neigh, leaf_label)
                if s < best_score:
                    best_score = s
                    best_children = neigh
                    best_label = lab

        if best_score < current_score:
            current_score = best_score
            current_children = best_children
            current_label = best_label
            results.append(str(current_score) + "\n" + format_tree(current_children, current_label))
        else:
            break

    return "\n\n".join(results)
